# FastAPI: Book Catalog API

This notebook builds and runs a small "Book Catalog" REST API end to end:
- `fastapi_utils.create_book_app()` builds the `FastAPI` app
- `fastapi_utils.run_server_in_background()` serves it with a real
  `uvicorn` server, on a background thread so the rest of the notebook can
  keep running
- `httpx` drives the app over real HTTP, the same way a browser or another
  service would

This is the production-shaped path. `fastapi.API.ipynb` covers the same
app's building blocks in-process with `TestClient`, which is faster for
exploration but skips the network entirely.

**Must run top to bottom after a kernel restart.**

In [ ]:
!pip install --quiet -r tutorial_requirements.txt

In [1]:
%load_ext autoreload
%autoreload 2

import logging

import httpx

import fastapi_utils

logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)

PORT = 8010
BASE_URL = f"http://127.0.0.1:{PORT}"

## Part 1: Build the App

`create_book_app()` wires up the routes and seeds an in-memory catalog.
Listing `app.routes` shows exactly what got registered, which is a useful
sanity check before starting the server.

In [2]:
app = fastapi_utils.create_book_app()

for route in app.routes:
    methods = ",".join(sorted(getattr(route, "methods", []) or []))
    _LOG.info("%-6s %s", methods, getattr(route, "path", route))

INFO:__main__:GET,HEAD /openapi.json


INFO:__main__:GET,HEAD /docs


INFO:__main__:GET,HEAD /docs/oauth2-redirect


INFO:__main__:GET,HEAD /redoc


INFO:__main__:GET    /health


INFO:__main__:GET    /books


INFO:__main__:GET    /books/{book_id}


INFO:__main__:POST   /books


INFO:__main__:PATCH  /books/{book_id}


INFO:__main__:DELETE /books/{book_id}


## Part 2: Start the Server

`run_server_in_background()` starts `uvicorn` on a daemon thread.
`wait_for_server()` polls `/health` until the socket accepts connections,
so the next cell never races the server startup.

In [3]:
server, server_thread = fastapi_utils.run_server_in_background(app, port=PORT)
fastapi_utils.wait_for_server(f"{BASE_URL}/health")
_LOG.info("Server is up at %s", BASE_URL)

INFO:fastapi_utils:Started uvicorn on 'http://127.0.0.1:8010' in a background thread.


INFO:httpx:HTTP Request: GET http://127.0.0.1:8010/health "HTTP/1.1 200 OK"


INFO:__main__:Server is up at http://127.0.0.1:8010


## Part 3: List and Filter Books

These are real HTTP requests: `httpx` opens a TCP connection to
`127.0.0.1:8010` instead of calling the app object directly.

In [4]:
response = httpx.get(f"{BASE_URL}/books")
response.raise_for_status()
_LOG.info("All books: %s", [book["title"] for book in response.json()])

response = httpx.get(f"{BASE_URL}/books", params={"in_stock": False})
_LOG.info("Out-of-stock books: %s", [book["title"] for book in response.json()])

response = httpx.get(f"{BASE_URL}/books", params={"limit": 1})
_LOG.info("First book only: %s", [book["title"] for book in response.json()])

INFO:httpx:HTTP Request: GET http://127.0.0.1:8010/books "HTTP/1.1 200 OK"


INFO:__main__:All books: ['Fluent Python', 'Clean Code', 'Designing Data-Intensive Applications']


INFO:httpx:HTTP Request: GET http://127.0.0.1:8010/books?in_stock=false "HTTP/1.1 200 OK"


INFO:__main__:Out-of-stock books: ['Designing Data-Intensive Applications']


INFO:httpx:HTTP Request: GET http://127.0.0.1:8010/books?limit=1 "HTTP/1.1 200 OK"


INFO:__main__:First book only: ['Fluent Python']


## Part 4: Create, Update, and Delete a Book

In [5]:
response = httpx.post(
    f"{BASE_URL}/books",
    json={"title": "The Pragmatic Programmer", "author": "Hunt & Thomas", "year": 1999},
)
response.raise_for_status()
new_book = response.json()
_LOG.info("Created book %s: %s", new_book["id"], new_book["title"])

INFO:httpx:HTTP Request: POST http://127.0.0.1:8010/books "HTTP/1.1 201 Created"


INFO:__main__:Created book 4: The Pragmatic Programmer


In [6]:
response = httpx.patch(
    f"{BASE_URL}/books/{new_book['id']}",
    json={
        "title": new_book["title"],
        "author": new_book["author"],
        "year": new_book["year"],
        "in_stock": False,
    },
)
response.raise_for_status()
_LOG.info("Updated book: %s", response.json())

INFO:httpx:HTTP Request: PATCH http://127.0.0.1:8010/books/4 "HTTP/1.1 200 OK"


INFO:__main__:Updated book: {'id': 4, 'title': 'The Pragmatic Programmer', 'author': 'Hunt & Thomas', 'year': 1999, 'in_stock': False}


In [7]:
response = httpx.delete(f"{BASE_URL}/books/{new_book['id']}")
_LOG.info("DELETE status: %s", response.status_code)

response = httpx.get(f"{BASE_URL}/books/{new_book['id']}")
_LOG.info("GET after delete: %s", response.status_code)

INFO:httpx:HTTP Request: DELETE http://127.0.0.1:8010/books/4 "HTTP/1.1 204 No Content"


INFO:__main__:DELETE status: 204


INFO:httpx:HTTP Request: GET http://127.0.0.1:8010/books/4 "HTTP/1.1 404 Not Found"


INFO:__main__:GET after delete: 404


## Part 5: Handle Errors

A missing book returns `404`; an invalid payload never reaches the handler
and returns `422` with a description of what failed.

In [8]:
response = httpx.get(f"{BASE_URL}/books/99999")
_LOG.info("GET missing book -> %s %s", response.status_code, response.json())

response = httpx.post(f"{BASE_URL}/books", json={"title": "No Author or Year"})
_LOG.info("POST invalid payload -> %s", response.status_code)
for error in response.json()["detail"]:
    _LOG.info("  %s: %s", error["loc"], error["msg"])

INFO:httpx:HTTP Request: GET http://127.0.0.1:8010/books/99999 "HTTP/1.1 404 Not Found"


INFO:__main__:GET missing book -> 404 {'detail': 'Book 99999 not found'}


INFO:httpx:HTTP Request: POST http://127.0.0.1:8010/books "HTTP/1.1 422 Unprocessable Entity"


INFO:__main__:POST invalid payload -> 422


INFO:__main__:  ['body', 'author']: Field required


INFO:__main__:  ['body', 'year']: Field required


## Part 6: Inspect the Live Docs

With the server running, `/openapi.json` is reachable over HTTP, and so are
the human-facing `/docs` (Swagger UI) and `/redoc` pages. Point a browser at
`http://127.0.0.1:8010/docs` while this notebook's server is up to try it
interactively.

In [9]:
response = httpx.get(f"{BASE_URL}/openapi.json")
schema = response.json()
_LOG.info("OpenAPI title: %s", schema["info"]["title"])
_LOG.info("Registered paths: %s", sorted(schema["paths"].keys()))

INFO:httpx:HTTP Request: GET http://127.0.0.1:8010/openapi.json "HTTP/1.1 200 OK"


INFO:__main__:OpenAPI title: Book Catalog API


INFO:__main__:Registered paths: ['/books', '/books/{book_id}', '/health']


## Part 7: Shut Down the Server

Always stop the background server before the kernel exits, so the port is
freed for the next run.

In [10]:
fastapi_utils.stop_server(server, server_thread)
_LOG.info("Server thread alive: %s", server_thread.is_alive())

INFO:fastapi_utils:Stopped background uvicorn server.


INFO:__main__:Server thread alive: False


## Wrap-up

This notebook covered the full lifecycle of a `FastAPI` service: build the
app, serve it with `uvicorn`, call it over real HTTP, and shut it down
cleanly. See `README.md` for how to run the same app from the command line
with `uvicorn fastapi_utils:app --reload` instead of from a notebook.